In [ ]:
# How do demographic factors shape listenings patterns
from pyspark.sql import SparkSession


# Part I — Music Artist Bridge Detection

## 1. Project Overview

**Objective:** Study whether collaborations between artists from different countries are associated with changes in listening activity.

### Data sources
- **ListenBrainz:** listening events, timestamps, artist names and MusicBrainz identifiers.
- **MusicBrainz:** artist metadata, geographical areas and artist relationships.

### Technologies
Python, PySpark, Spark SQL and Parquet.

### Workflow
1. Explore raw datasets.
2. Define schemas and ingest the data.
3. Convert the data into typed Parquet tables.
4. Clean and validate the data.
5. Integrate the datasets.
6. Analyse listening activity and international collaborations.


In [1]:
from pathlib import Path
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("ArtistBridge") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

RAW = Path("/home/jovyan/work/data/raw/mb")
OUT = Path("/home/jovyan/work/data/processed")
CLEAN = Path("/home/jovyan/work/data/clean")

OUT.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

MB = RAW / "mbdump"

In [2]:
schema = StructType([
    StructField("listened_at", TimestampType()),
    StructField("created", TimestampType()),
    StructField("user_id", LongType()),
    StructField("recording_msid", StringType()),
    StructField("artist_name", StringType()),
    StructField("artist_credit_id", LongType()),
    StructField("release_name", StringType()),
    StructField("release_mbid", StringType()),
    StructField("recording_name", StringType()),
    StructField("recording_mbid", StringType()),
    StructField("artist_credit_mbids", ArrayType(StringType()))
])

LB_FILES = [
    RAW / "listenbrainz-spark-dump-2702-20261006-000002-incremental",
    RAW / "listenbrainz-spark-dump-2704-20261007-000002-incremental",
    RAW / "listenbrainz-spark-dump-2706-20261008-000002-incremental"
]

In [3]:
shutil.rmtree(OUT / "listenbrainz", ignore_errors=True)

for p in LB_FILES:
    df = spark.read.schema(schema).parquet(str(p / "*.parquet"))
    df = df.withColumn("dump_source", F.lit(p.name))

    df.write.mode("append") \
        .option("parquet.enable.dictionary", "false") \
        .parquet(str(OUT / "listenbrainz"))

In [4]:
lb = spark.read.parquet(str(OUT / "listenbrainz"))

print("Lignes :", lb.count())
lb.printSchema()
lb.groupBy("dump_source").count().show(truncate=False)

Lignes : 11459693
root
 |-- listened_at: timestamp (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- recording_msid: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_credit_id: long (nullable = true)
 |-- release_name: string (nullable = true)
 |-- release_mbid: string (nullable = true)
 |-- recording_name: string (nullable = true)
 |-- recording_mbid: string (nullable = true)
 |-- artist_credit_mbids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dump_source: string (nullable = true)

+--------------------------------------------------------+-------+
|dump_source                                             |count  |
+--------------------------------------------------------+-------+
|listenbrainz-spark-dump-2704-20261007-000002-incremental|2978911|
|listenbrainz-spark-dump-2702-20261006-000002-incremental|4212278|
|listenbrainz-spark-dump-2706-20261008-000002-incremental|42

In [5]:
artist_schema = StructType([
    StructField("id", LongType()),
    StructField("gid", StringType()),
    StructField("name", StringType()),
    StructField("sort_name", StringType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("type", IntegerType()),
    StructField("area", LongType()),
    StructField("gender", IntegerType()),
    StructField("comment", StringType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", TimestampType()),
    StructField("ended", StringType()),
    StructField("begin_area", LongType()),
    StructField("end_area", LongType())
])

area_schema = StructType([
    StructField("id", LongType()),
    StructField("gid", StringType()),
    StructField("name", StringType()),
    StructField("type", IntegerType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", TimestampType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("ended", StringType()),
    StructField("comment", StringType())
])

link_schema = StructType([
    StructField("id", LongType()),
    StructField("link_type", LongType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("attribute_count", IntegerType()),
    StructField("created", TimestampType()),
    StructField("ended", StringType())
])

link_type_schema = StructType([
    StructField("id", LongType()),
    StructField("parent", LongType()),
    StructField("child_order", IntegerType()),
    StructField("gid", StringType()),
    StructField("entity_type0", StringType()),
    StructField("entity_type1", StringType()),
    StructField("name", StringType()),
    StructField("description", StringType()),
    StructField("link_phrase", StringType()),
    StructField("reverse_link_phrase", StringType()),
    StructField("long_link_phrase", StringType()),
    StructField("last_updated", TimestampType()),
    StructField("deprecated", StringType()),
    StructField("has_dates", StringType()),
    StructField("entity0_cardinality", IntegerType()),
    StructField("entity1_cardinality", IntegerType())
])

l_artist_artist_schema = StructType([
    StructField("id", LongType()),
    StructField("link", LongType()),
    StructField("entity0", LongType()),
    StructField("entity1", LongType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", TimestampType()),
    StructField("link_order", IntegerType()),
    StructField("entity0_credit", LongType()),
    StructField("entity1_credit", LongType())
])

In [8]:
schemas = {
    "artist": artist_schema,
    "area": area_schema,
    "link": link_schema,
    "link_type": link_type_schema,
    "l_artist_artist": l_artist_artist_schema,
}

bool_cols = {
    "artist": ["ended"],
    "area": ["ended"],
    "link": ["ended"],
    "link_type": ["deprecated", "has_dates"],
    "l_artist_artist": [],
}

for name, sch in schemas.items():
    df = spark.read.schema(sch) \
        .option("sep", "\t") \
        .option("header", "false") \
        .option("nullValue", r"\N") \
        .csv(str(MB / name))

    for c in bool_cols[name]:
        df = df.withColumn(
            c,
            F.when(F.col(c) == "t", True)
             .when(F.col(c) == "f", False)
             .otherwise(None)
        )

    df.write.mode("overwrite").parquet(str(OUT / f"mb_{name}"))

In [9]:
for name in schemas:
    df = spark.read.parquet(str(OUT / f"mb_{name}"))
    print(name, df.count())
    df.printSchema()

artist 3015712
root
 |-- id: long (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sort_name: string (nullable = true)
 |-- begin_date_year: integer (nullable = true)
 |-- begin_date_month: integer (nullable = true)
 |-- begin_date_day: integer (nullable = true)
 |-- end_date_year: integer (nullable = true)
 |-- end_date_month: integer (nullable = true)
 |-- end_date_day: integer (nullable = true)
 |-- type: integer (nullable = true)
 |-- area: long (nullable = true)
 |-- gender: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- ended: boolean (nullable = true)
 |-- begin_area: long (nullable = true)
 |-- end_area: long (nullable = true)

area 120108
root
 |-- id: long (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- type: integer (nullable = true)
 |-- edits_pending: integer (nullable 

In [10]:
for name in ["artist", "area", "link", "link_type", "l_artist_artist"]:
    df = spark.read.parquet(str(OUT / f"mb_{name}"))
    dates = [c for c in ["last_updated", "created"] if c in df.columns]

    for c in dates:
        print(name, c, df.filter(F.col(c).isNull()).count())

artist last_updated 136247
area last_updated 0
link created 0
link_type last_updated 0
l_artist_artist last_updated 0


In [11]:
lb = spark.read.parquet(str(OUT / "listenbrainz"))
artist = spark.read.parquet(str(OUT / "mb_artist"))
area = spark.read.parquet(str(OUT / "mb_area"))
link = spark.read.parquet(str(OUT / "mb_link"))
link_type = spark.read.parquet(str(OUT / "mb_link_type"))
l_artist_artist = spark.read.parquet(str(OUT / "mb_l_artist_artist"))


In [12]:
lb.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in lb.columns
]).show(truncate=False)

artist.select(
    F.sum(F.col("area").isNull().cast("long")).alias("area_nulls")
).show()

+-----------+-------+-------+--------------+-----------+----------------+------------+------------+--------------+--------------+-------------------+-----------+
|listened_at|created|user_id|recording_msid|artist_name|artist_credit_id|release_name|release_mbid|recording_name|recording_mbid|artist_credit_mbids|dump_source|
+-----------+-------+-------+--------------+-----------+----------------+------------+------------+--------------+--------------+-------------------+-----------+
|0          |0      |0      |0             |0          |2146861         |297380      |2148400     |0             |2100611       |2001475            |0          |
+-----------+-------+-------+--------------+-----------+----------------+------------+------------+--------------+--------------+-------------------+-----------+

+----------+
|area_nulls|
+----------+
|   1518240|
+----------+



In [13]:
for name in ["artist", "area", "link", "link_type", "l_artist_artist"]:
    df = spark.read.parquet(str(OUT / f"mb_{name}"))

    print(name, "lignes :", df.count())
    print("IDs distincts :", df.select("id").distinct().count())

    if "gid" in df.columns:
        print("GID distincts :", df.select("gid").distinct().count())

artist lignes : 3015712
IDs distincts : 3015712
GID distincts : 3015712
area lignes : 120108
IDs distincts : 120108
GID distincts : 120108
link lignes : 1155021
IDs distincts : 1155021
link_type lignes : 697
IDs distincts : 697
GID distincts : 697
l_artist_artist lignes : 865360
IDs distincts : 865360


In [14]:
uuid = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"

for c in ["recording_mbid", "release_mbid"]:
    print(c, lb.filter(
        F.col(c).isNotNull() & ~F.col(c).rlike(uuid)
    ).count())

lb = lb.withColumn(
    "recording_mbid", F.lower("recording_mbid")
).withColumn(
    "release_mbid", F.lower("release_mbid")
).withColumn(
    "artist_credit_mbids",
    F.transform("artist_credit_mbids", lambda x: F.lower(x))
)

artist = artist.withColumn("gid", F.lower("gid"))
area = area.withColumn("gid", F.lower("gid"))
link_type = link_type.withColumn("gid", F.lower("gid"))

recording_mbid 0
release_mbid 0


In [15]:
mbids = lb.select(
    F.explode("artist_credit_mbids").alias("mbid")
)

print("MBID non UUID :", mbids.filter(
    F.col("mbid").isNotNull() & ~F.col("mbid").rlike(uuid)
).count())

MBID non UUID : 0


In [16]:
print("Écoutes futures :", lb.filter(
    F.col("listened_at") > F.current_timestamp()
).count())

print("Écoutes avant 1900 :", lb.filter(
    F.col("listened_at") < F.to_timestamp(F.lit("1900-01-01"))
).count())

Écoutes futures : 0
Écoutes avant 1900 : 0


In [17]:
lb = lb.withColumn(
    "track_key",
    F.coalesce(
        "recording_msid",
        "recording_mbid",
        F.when(
            F.col("recording_name").isNotNull(),
            F.concat_ws(" - ", "artist_name", "recording_name")
        )
    )
)

avant_doublons = lb.count()

known = lb.filter(F.col("track_key").isNotNull())
unknown = lb.filter(F.col("track_key").isNull())

known = known.dropDuplicates([
    "user_id", "listened_at", "track_key"
])

lb = known.unionByName(unknown)

apres_doublons = lb.count()

print("Avant :", avant_doublons)
print("Après :", apres_doublons)
print("Doublons retirés :", avant_doublons - apres_doublons)

Avant : 11459693
Après : 11459693
Doublons retirés : 0


In [18]:
lb.groupBy(
    "user_id", "listened_at", "recording_msid"
).count().filter(
    F.col("count") > 1
).agg(
    F.sum(F.col("count") - 1).alias("doublons")
).show()

+--------+
|doublons|
+--------+
|    NULL|
+--------+



In [19]:
artist_raw = spark.read.option("sep", "\t") \
    .option("header", "false") \
    .option("nullValue", r"\N") \
    .csv(str(MB / "artist"))

link_type_raw = spark.read.option("sep", "\t") \
    .option("header", "false") \
    .option("nullValue", r"\N") \
    .csv(str(MB / "link_type"))

artist_raw.select(
    F.count("*").alias("lignes"),
    F.sum(F.col("_c15").isNull().cast("long")).alias("dates_nulles_source")
).show()

link_type_raw.select(
    F.count("*").alias("lignes"),
    F.sum(F.col("_c11").isNull().cast("long")).alias("dates_nulles_source")
).show()

+-------+-------------------+
| lignes|dates_nulles_source|
+-------+-------------------+
|3015712|             136245|
+-------+-------------------+

+------+-------------------+
|lignes|dates_nulles_source|
+------+-------------------+
|   697|                  0|
+------+-------------------+



In [20]:
artist_raw.select("_c15").show(5, False)
link_type_raw.select("_c11").show(5, False)

+-----------------------------+
|_c15                         |
+-----------------------------+
|2021-11-23 07:08:52.479537+00|
|NULL                         |
|2025-09-14 23:53:28.004798+00|
|2024-10-19 03:33:55.474151+00|
|NULL                         |
+-----------------------------+
only showing top 5 rows

+-----------------------------+
|_c11                         |
+-----------------------------+
|2014-05-18 09:46:23.72719+00 |
|2014-05-18 10:41:05.403719+00|
|2015-12-16 11:53:10.160133+00|
|2018-05-18 10:37:46.114037+00|
|2021-02-02 10:14:29.683106+00|
+-----------------------------+
only showing top 5 rows



In [21]:
lb.filter(F.col("recording_msid").isNotNull()) \
  .groupBy("user_id", "listened_at", "recording_msid") \
  .count() \
  .filter(F.col("count") > 1) \
  .agg(F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("doublons")) \
  .show()

+--------+
|doublons|
+--------+
|       0|
+--------+



In [22]:
link_type_raw.select(
    "_c0", "_c1", "_c9", "_c10", "_c11", "_c12", "_c13", "_c14", "_c15"
).show(5, False)

+---+----+------------------------+------------------------+-----------------------------+----+----+----+----+
|_c0|_c1 |_c9                     |_c10                    |_c11                         |_c12|_c13|_c14|_c15|
+---+----+------------------------+------------------------+-----------------------------+----+----+----+----+
|12 |NULL|covers or other versions|covers and versions     |2014-05-18 09:46:23.72719+00 |f   |f   |0   |0   |
|735|NULL|child of                |has child               |2014-05-18 10:41:05.403719+00|f   |f   |0   |0   |
|870|784 |SoundCloud page for     |has a SoundCloud page at|2015-12-16 11:53:10.160133+00|f   |t   |0   |0   |
|957|NULL|get the music           |get the music           |2018-05-18 10:37:46.114037+00|f   |f   |0   |0   |
|227|234 |DJ-mixes                |is a DJ-mix of          |2021-02-02 10:14:29.683106+00|f   |t   |0   |0   |
+---+----+------------------------+------------------------+-----------------------------+----+----+----+----+
o

In [23]:
link_type_raw.select("_c12", "_c13", "_c14", "_c15").printSchema()

root
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)



In [24]:
link_type_raw.show(3, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------
 _c0  | 12                                                                                                                    
 _c1  | NULL                                                                                                                  
 _c2  | 0                                                                                                                     
 _c3  | 38278b3b-30e6-304c-b0db-5ba701eb0268                                                                                  
 _c4  | release_group                                                                                                         
 _c5  | release_group                                                                                                         
 _c6  | covers and versions                                                                                    

In [25]:
user_counts = lb.groupBy("user_id").count()

user_counts.orderBy(F.desc("count")).show(20)

bots = user_counts.filter(F.col("count") > 10000)

print("Utilisateurs au-dessus du seuil :", bots.count())

+-------+------+
|user_id| count|
+-------+------+
| 208146|453597|
| 207524|253025|
| 207751|248363|
| 205515|240675|
| 206491|223318|
| 206680|201884|
| 203494|182879|
| 206767|176240|
| 204561|171427|
| 182587|169823|
| 204069|168982|
| 207993|166548|
| 206475|155893|
| 207664|155378|
| 204667|149548|
| 206516|146072|
| 205682|141340|
| 206458|138267|
| 179530|129531|
| 205557|116170|
+-------+------+
only showing top 20 rows

Utilisateurs au-dessus du seuil : 93


In [26]:
lb = lb.filter(
    F.col("listened_at").isNotNull() &
    (F.col("listened_at") <= F.current_timestamp()) &
    (F.col("listened_at") >= F.to_timestamp(F.lit("1900-01-01")))
)
n_dates = lb.count()

lb = lb.join(
    bots.select("user_id"),
    on="user_id",
    how="left_anti"
)
n_bots = lb.count()

n_artist_avant = artist.count()
artist = artist.filter(
    F.col("name").isNotNull() &
    (F.trim(F.col("name")) != "")
)
n_artist_apres = artist.count()

print(apres_doublons, n_dates, n_bots, n_artist_avant, n_artist_apres)

11459693 11459693 4345435 3015712 3015712


In [27]:
lb.drop("track_key").write.mode("overwrite").parquet(
    str(CLEAN / "listenbrainz")
)

artist.write.mode("overwrite").parquet(str(CLEAN / "artist"))
area.write.mode("overwrite").parquet(str(CLEAN / "area"))
link.write.mode("overwrite").parquet(str(CLEAN / "link"))
link_type.write.mode("overwrite").parquet(str(CLEAN / "link_type"))
l_artist_artist.write.mode("overwrite").parquet(
    str(CLEAN / "l_artist_artist")
)


In [28]:
lb_clean = spark.read.parquet(str(CLEAN / "listenbrainz"))
artist_clean = spark.read.parquet(str(CLEAN / "artist"))
n_final = lb_clean.count()

recap = [
    ("Doublons", avant_doublons, apres_doublons,
     "Même utilisateur, date et morceau"),
    ("Dates invalides", apres_doublons, n_dates,
     "Dates futures ou antérieures à 1900"),
    ("Utilisateurs suspects", n_dates, n_bots,
     "Seuil de 10000 écoutes"),
    ("Artistes sans nom", n_artist_avant, n_artist_apres,
     "Nom nul ou vide"),
    ("artist_credit_id manquant", n_final, n_final,
     "Conservé : la jointure utilise les MBID"),
    ("recording_mbid manquant", n_final, n_final,
     "Conservé : non nécessaire à la jointure principale"),
    ("area manquant", n_artist_apres, n_artist_apres,
     "Conservé : pays inconnu pour ces artistes")
]

recap_df = spark.createDataFrame(
    recap,
    ["règle", "lignes avant", "lignes après", "raison"]
)

recap_df.show(truncate=False)

+-------------------------+------------+------------+--------------------------------------------------+
|règle                    |lignes avant|lignes après|raison                                            |
+-------------------------+------------+------------+--------------------------------------------------+
|Doublons                 |11459693    |11459693    |Même utilisateur, date et morceau                 |
|Dates invalides          |11459693    |11459693    |Dates futures ou antérieures à 1900               |
|Utilisateurs suspects    |11459693    |4345435     |Seuil de 10000 écoutes                            |
|Artistes sans nom        |3015712     |3015712     |Nom nul ou vide                                   |
|artist_credit_id manquant|4345435     |4345435     |Conservé : la jointure utilise les MBID           |
|recording_mbid manquant  |4345435     |4345435     |Conservé : non nécessaire à la jointure principale|
|area manquant            |3015712     |3015712     |Co

In [29]:
lb_clean = spark.read.parquet(str(CLEAN / "listenbrainz"))
artist_clean = spark.read.parquet(str(CLEAN / "artist"))
area_clean = spark.read.parquet(str(CLEAN / "area"))

In [30]:
lb_ids = lb_clean.withColumn(
    "listen_id", F.monotonically_increasing_id()
).withColumn(
    "artist_mbid", F.explode("artist_credit_mbids")
).filter(
    F.col("artist_mbid").isNotNull()
)

print("Écoutes avant jointure :", lb_clean.count())

Écoutes avant jointure : 4345435


In [31]:
total = lb_clean.count()
with_artist = lb_clean.filter(F.size("artist_credit_mbids") > 0).count()

lb_artist = lb_ids.join(
    artist_clean.select(
        "gid",
        F.col("name").alias("mb_name"),
        "area"
    ),
    lb_ids.artist_mbid == artist_clean.gid,
    "inner"
)

matched = lb_artist.select("listen_id").distinct().count()

print("Écoutes retrouvées dans MusicBrainz :", matched)
print("Taux (sur toutes les écoutes) :", round(matched / total * 100, 2), "%")
print("Taux (sur écoutes avec MBID) :", round(matched / with_artist * 100, 2), "%")

Écoutes retrouvées dans MusicBrainz : 3474923
Taux (sur toutes les écoutes) : 79.97 %
Taux (sur écoutes avec MBID) : 99.99 %


In [32]:
countries = area_clean.filter(
    F.col("type") == 1
).select(
    F.col("id").alias("area_id"),
    F.col("name").alias("country")
)

lb_country = lb_artist.join(
    countries,
    lb_artist.area == countries.area_id,
    "inner"
)

country_matched = lb_country.select("listen_id").distinct().count()

print("Écoutes associées à un pays :", country_matched)
print("Taux de correspondance pays :", round(
    country_matched / lb_clean.count() * 100, 2
), "%")

Écoutes associées à un pays : 2988540
Taux de correspondance pays : 68.77 %


In [33]:
country_stats = lb_country.groupBy("country").agg(
    F.countDistinct("listen_id").alias("ecoutes"),
    F.countDistinct("user_id").alias("utilisateurs")
)

country_stats.orderBy(F.desc("ecoutes")).show(20, truncate=False)
country_stats.orderBy(F.desc("utilisateurs")).show(20, truncate=False)

+--------------+-------+------------+
|country       |ecoutes|utilisateurs|
+--------------+-------+------------+
|United States |1078671|25551       |
|South Korea   |809905 |3992        |
|United Kingdom|340157 |19803       |
|Japan         |137709 |7567        |
|Canada        |95954  |12708       |
|Germany       |91968  |8840        |
|France        |69473  |8750        |
|Sweden        |56625  |8274        |
|Australia     |53960  |9620        |
|Italy         |24997  |4301        |
|Netherlands   |22401  |4283        |
|Russia        |22345  |2633        |
|Brazil        |19965  |2321        |
|Norway        |18657  |3911        |
|Ireland       |16590  |3942        |
|Spain         |16562  |2522        |
|Finland       |16515  |2532        |
|Belgium       |12205  |2706        |
|Mexico        |10936  |1653        |
|Puerto Rico   |10125  |1591        |
+--------------+-------+------------+
only showing top 20 rows

+--------------+-------+------------+
|country       |ecoutes|

In [34]:
user_countries = lb_country.groupBy("user_id").agg(
    F.countDistinct("country").alias("nb_pays")
)

user_countries.orderBy(F.desc("nb_pays")).show(20)

+-------+-------+
|user_id|nb_pays|
+-------+-------+
|  89849|     62|
| 208325|     60|
| 208154|     51|
| 207470|     49|
| 206648|     47|
| 207806|     46|
| 207089|     46|
|  71134|     45|
|  94168|     43|
| 207341|     43|
| 206535|     42|
|  13575|     41|
| 106580|     41|
| 114146|     41|
| 181986|     41|
| 125364|     40|
|  43361|     40|
| 142894|     40|
|   3569|     39|
|  90803|     39|
+-------+-------+
only showing top 20 rows



In [35]:
artist_stats = lb_artist.groupBy("gid").agg(
    F.first("mb_name").alias("artist_name"),
    F.countDistinct("listen_id").alias("ecoutes")
)

artist_stats.orderBy(F.desc("ecoutes")).show(20, truncate=False)

+------------------------------------+--------------+-------+
|gid                                 |artist_name   |ecoutes|
+------------------------------------+--------------+-------+
|0d79fe8e-ba27-4859-bb8c-2f255f346853|BTS           |565257 |
|f09d2950-e3c6-47b2-b21c-2bad2cd3f616|Agust D       |43879  |
|d2582c77-6b0f-4b2f-802f-8f3cad1a5fec|Jimin         |32164  |
|58f1e354-ecbc-4fd3-aa25-97eb1d9bd8b9|RM            |25622  |
|23c8056b-ee13-4cfc-a772-2f5292e35bb5|Jung Kook     |24764  |
|b3977da4-f899-4328-921b-ecc599a5618a|j‐hope        |16977  |
|83096042-3785-481e-8843-dee69f1aad12|V             |16309  |
|09720eec-3871-49d5-932d-eb7542768cd3|Jin           |15074  |
|f59c5520-5f46-4d2c-b2c4-822eabf53419|Linkin Park   |14078  |
|20244d07-534f-4eff-b4d4-930878889970|Taylor Swift  |10592  |
|d27808c8-d3a6-4bbe-acc2-22d5e19f3bca|CORTIS        |10341  |
|a74b1b7f-71a5-4011-9441-d0b5e4122711|Radiohead     |9529   |
|355cc194-b8db-4af2-8678-c7ad54213f4b|Latto         |9332   |
|b10bbbf

In [36]:
check = lb_clean.filter(
    F.size("artist_credit_mbids") == 1
).withColumn(
    "artist_mbid", F.explode("artist_credit_mbids")
).join(
    artist_clean,
    F.col("artist_mbid") == F.col("gid"),
    "inner"
).select(
    "artist_name",
    F.col("name").alias("musicbrainz_name"),
    "artist_mbid"
)

check.show(20, truncate=False)

+-----------------+-----------------+------------------------------------+
|artist_name      |musicbrainz_name |artist_mbid                         |
+-----------------+-----------------+------------------------------------+
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway           |0001cd84-2a11-4699-8d6b-0abf969c5f06|
|Midway           |Midway

In [37]:
country_stats.write.mode("overwrite").parquet(
    str(CLEAN / "country_stats")
)

user_countries.write.mode("overwrite").parquet(
    str(CLEAN / "user_countries")
)

artist_stats.write.mode("overwrite").parquet(
    str(CLEAN / "artist_stats")
)

In [38]:
from pathlib import Path

for path in [CLEAN / "listenbrainz"]:
    size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    print(f"Taille : {size / (1024**3):.2f} Go")

Taille : 0.49 Go
